# Đồ án 2: Data Fitting và Phương pháp OLS
## Phần 2: Ứng dụng Data Fitting vào Dữ liệu Thực tế

**Nhóm:** 24CTT3 - Nhóm 2 

**Thành viên:** 
- Ngô Hoàng Minh - 24120381 
- Mai Thúc Hải Đăng  - 24120276       
- Nguyễn Thành Dự - 24120288 
- Đỗ Ngọc Hải - 24120300 
- Nguyễn Xuân Lộc - 24120369

### Mục tiêu:
1. Thực hiện Khảo sát dữ liệu (EDA) cho bộ dữ liệu Taxi Trip Pricing.
2. Xây dựng Pipeline tiền xử lý hoàn chỉnh (xử lý missing, outlier, encoding, feature engineering).
3. Huấn luyện và so sánh các mô hình hồi quy: OLS (Full & Selected), Ridge, Lasso.
4. Triển khai các kỹ thuật nâng cao: Kernel Ridge và Bayesian Linear Regression.
5. Đánh giá và nhận xét hiệu năng tổng thể.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Thêm gốc dự án vào path để import các module local
sys.path.append(os.path.abspath('..'))

from Part1.ols_implementation import ols_fit, model_metrics, calculate_vif
from Part1.ridge_lasso import ridge_fit, lasso_fit
from Part1.cross_validation import kfold_cv
from Part1.helper_function import add_intercept, matvec, matmul

from Part2.data_pipeline import DataPipeline
import Part2.model_comparison as mc
from Part2.model_comparison import (
    fit_ols, predict_ols, calculate_metrics, 
    select_features_vif, select_features_pvalue, 
    plot_residual_diagnostics, plot_feature_importance
)
from Part2.advanced_methods import KernelRidgeRegression, BayesianLinearRegression

# VÁ LỖI: File model_comparison.py của nhóm đang thiếu import plt và sns
# Chúng ta sẽ tiêm (inject) các thư viện này vào namespace của module đó
mc.plt = plt
mc.sns = sns

# Cố định random state để đảm bảo tính tái lập (Reproducible)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import warnings
warnings.filterwarnings('ignore')

print("Thiết lập hoàn tất. (Đã vá lỗi thiếu import trong model_comparison.py)")

## 1. Khảo sát dữ liệu (Exploratory Data Analysis - EDA)
Trong phần này, chúng ta sẽ thực hiện:
* Thống kê mô tả các biến số.
* Vẽ biểu đồ phân phối (Histogram/Boxplot) để phát hiện xu hướng và outlier.
* Phân tích tương quan giữa các biến.
* Phân tích tỉ lệ dữ liệu thiếu (Missing Values).

In [ ]:
# Khởi tạo Pipeline để sử dụng hàm EDA tích hợp sẵn
pipeline_eda = DataPipeline()
if pipeline_eda.df is not None:
    pipeline_eda.EDA()
else:
    print("Lỗi: Không tìm thấy file dữ liệu taxi_trip_pricing.csv")

## 2. Tiền xử lý dữ liệu (Data Preprocessing)
### Lý luận khoa học:
* **Cơ chế thiếu dữ liệu:** Dựa trên quan sát, dữ liệu thiếu khoảng 5% ở hầu hết các cột. Chúng tôi giả định đây là cơ chế **MAR (Missing At Random)** và sử dụng **KNN Imputation** để tận dụng thông tin từ các biến khác nhằm điền khuyết chính xác hơn là dùng Mean đơn thuần.
* **Xử lý Outlier:** Sử dụng phương pháp **Winsorization** (clip giá trị tại ngưỡng 1.5*IQR) để giảm thiểu ảnh hưởng của các chuyến đi có giá hoặc quãng đường cực đoan mà không làm mất mẫu dữ liệu.
* **Feature Engineering:** Tạo thêm biến **Log transformation** cho `Trip_Distance_km` để xử lý độ lệch (skewness) và **Polynomial features (bậc 2)** để bắt kịp các mối quan hệ phi tuyến tiềm năng.

In [ ]:
from sklearn.model_selection import train_test_split

# Đọc dữ liệu gốc
data_path = os.path.join('data', 'taxi_trip_pricing.csv')
df_raw = pd.read_csv(data_path)
df_clean = df_raw.dropna(subset=['Trip_Price']).reset_index(drop=True)

# Chia tập Train/Test (80/20)
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=RANDOM_STATE)

# Cấu hình Pipeline tiền xử lý
pipeline = DataPipeline(
    imputation_method='knn', 
    handle_outliers='winsorize',
    log_transform_cols=['Trip_Distance_km'],
    poly_degree=2,
    target_col='Trip_Price'
)

# Huấn luyện pipeline trên tập Train và biến đổi cả hai tập
X_train, y_train = pipeline.fit_transform(train_df)
X_test, y_test = pipeline.transform(test_df)

print(f"Kích thước tập huấn luyện: {X_train.shape}")
print(f"Danh sách các đặc trưng sau xử lý:\n{X_train.columns.tolist()}")

## 3. Xây dựng Mô hình cơ sở (OLS)
Chúng ta sẽ huấn luyện 2 biến thể:
1. **OLS Full:** Sử dụng toàn bộ đặc trưng sau khi Feature Engineering.
2. **OLS Selected:** Loại bỏ đa cộng tuyến bằng VIF và chọn biến có ý nghĩa bằng P-value.

In [ ]:
# 3.1 Hồi quy OLS Đầy đủ
beta_full = fit_ols(X_train, y_train)
y_pred_full = predict_ols(X_test, beta_full)
metrics_full = calculate_metrics(y_test, y_pred_full)

print("--- Kết quả OLS Full ---")
for m, v in metrics_full.items(): print(f"{m}: {v:.4f}")

In [ ]:
# 3.2 OLS Chọn biến (Dựa trên VIF & P-value)
print("\n--- Đang thực hiện quy trình chọn biến ---")
selected_vif = select_features_vif(X_train, threshold=5.0)
final_features = select_features_pvalue(X_train[selected_vif], y_train, alpha=0.05)

beta_sel = fit_ols(X_train[final_features], y_train)
y_pred_sel = predict_ols(X_test[final_features], beta_sel)
metrics_sel = calculate_metrics(y_test, y_pred_sel)

print(f"\nSố biến còn lại sau khi lọc: {len(final_features)}")
print("--- Kết quả OLS Selected ---")
for m, v in metrics_sel.items(): print(f"{m}: {v:.4f}")

## 4. Chính quy hóa (Ridge & Lasso)
Sử dụng 5-Fold Cross-Validation để tìm siêu tham số $\lambda$ tối ưu cho Ridge và Lasso.

In [ ]:
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0]
X_train_bias = add_intercept(X_train[final_features].values.tolist())
X_test_bias = add_intercept(X_test[final_features].values.tolist())
y_train_list = y_train.tolist()

# Hàm tìm lambda tốt nhất dựa trên MSE trung bình từ Cross-Validation
def find_best_lambda(model_func):
    best_lam, min_mse = None, float('inf')
    for lam in lambdas:
        mse, _ = kfold_cv(X_train_bias, y_train_list, k=5, model_func=model_func, lam=lam, fit_intercept=True)
        if mse < min_mse: min_mse, best_lam = mse, lam
    return best_lam

best_ridge_lam = find_best_lambda(ridge_fit)
best_lasso_lam = find_best_lambda(lasso_fit)

print(f"Lambda tối ưu cho Ridge: {best_ridge_lam}")
print(f"Lambda tối ưu cho Lasso: {best_lasso_lam}")

# Huấn luyện lại mô hình với lambda tốt nhất
beta_ridge = ridge_fit(X_train_bias, y_train_list, lam=best_ridge_lam, fit_intercept=True)
y_pred_ridge = matvec(X_test_bias, beta_ridge)
metrics_ridge = calculate_metrics(y_test, y_pred_ridge)

beta_lasso = lasso_fit(X_train_bias, y_train_list, lam=best_lasso_lam, fit_intercept=True)
y_pred_lasso = matvec(X_test_bias, beta_lasso)
metrics_lasso = calculate_metrics(y_test, y_pred_lasso)

## 5. Kỹ thuật nâng cao (Kernel Ridge & Bayesian)
Sử dụng các lớp đã được cài đặt từ đầu (from scratch) trong `advanced_methods.py`.

In [ ]:
# 5.1 Kernel Ridge Regression (Sử dụng RBF Kernel)
krr = KernelRidgeRegression(lmbda=1.0, length_scale=1.0)
krr.fit(X_train.values, y_train.values)
y_pred_krr = krr.predict(X_test.values)
metrics_krr = calculate_metrics(y_test, y_pred_krr)

# 5.2 Bayesian Linear Regression
blr = BayesianLinearRegression(sigma_sq=1.0)
X_train_bias_np = np.array(X_train_bias)
X_test_bias_np = np.array(X_test_bias)
blr.fit(X_train_bias_np, y_train.values) # Sử dụng ma trận X có cột intercept dưới dạng numpy
y_pred_blr = blr.predict(X_test_bias_np)
metrics_blr = calculate_metrics(y_test, y_pred_blr)

print("Huấn luyện các phương pháp nâng cao hoàn tất.")

## 6. Tổng hợp kết quả & Chẩn đoán mô hình
Cuối cùng, chúng ta sẽ xem xét bảng so sánh tổng hợp và vẽ các biểu đồ chẩn đoán cho mô hình baseline.

In [ ]:
# Tạo bảng tổng hợp các chỉ số đánh giá
summary_results = pd.DataFrame({
    'OLS Full': metrics_full,
    'OLS Selected': metrics_sel,
    'Ridge': metrics_ridge,
    'Lasso': metrics_lasso,
    'Kernel Ridge': metrics_krr,
    'Bayesian LR': metrics_blr
}).T
print("BẢNG TỔNG HỢP KẾT QUẢ TRÊN TẬP TEST")
print(summary_results)

### Nhận xét kết quả:
* **So sánh các mô hình:** Dựa trên bảng trên, chúng ta có thể xác định mô hình tối ưu bằng cách tìm giá trị **MAE, RMSE thấp nhất** và **$R^2$ cao nhất**. Thông thường, các mô hình có Regularization (Ridge/Lasso) hoặc chọn biến (OLS Selected) sẽ có sai số trên tập Test ổn định hơn do tránh được hiện tượng Overfitting.
* **Hiệu quả của phương pháp nâng cao:** Quan sát kết quả của **Kernel Ridge** và **Bayesian LR**. Nếu dữ liệu có mối quan hệ phi tuyến phức tạp, Kernel Ridge (với RBF kernel) thường sẽ cho $R^2$ vượt trội so với OLS thuần túy. Trong khi đó, Bayesian LR cung cấp kết quả tương đương OLS nhưng có thêm khả năng đánh giá độ tin cậy của dự báo (uncertainty quantification).

In [ ]:
print("TIẾN HÀNH TRỰC QUAN HÓA CHẨN ĐOÁN")
# Vẽ 4 biểu đồ chẩn đoán phần dư cho mô hình OLS Đầy đủ
plot_residual_diagnostics(X_test.values, y_test, y_pred_full, X_test.columns.tolist())

# Vẽ biểu đồ tầm quan trọng của các đặc trưng (Feature Importance)
plot_feature_importance(beta_full, X_test.columns.tolist())

#### 3. Tổng kết chung:

Để có một mô hình dự báo giá vé chính xác, việc làm sạch và chuẩn bị dữ liệu kỹ lưỡng (như xử lý biến thiếu bằng KNN hay giới hạn outlier) quan trọng không kém gì việc chọn thuật toán. Việc xây dựng một Pipeline hoàn chỉnh giúp quá trình thử nghiệm diễn ra nhanh chóng, khoa học và tránh được các sai sót về rò rỉ dữ liệu giữa tập Train và Test.